# Deepfake Detection — Training & Evaluation Only
**Preprocessing already done. This notebook fetches preprocessed data from Drive, trains models, evaluates, and uploads results.**

---
## 1. Environment Setup

In [ ]:
import time as _time
_NB_START = _time.time()
_STEP_START = _time.time()
_EST_TOTAL = 70 * 60
def _T(lbl=""):
    global _STEP_START
    elapsed = _time.time() - _NB_START
    step = _time.time() - _STEP_START
    remaining = max(0, _EST_TOTAL - elapsed)
    h, m, s = int(elapsed//3600), int((elapsed%3600)//60), int(elapsed%60)
    r_h, r_m, r_s = int(remaining//3600), int((remaining%3600)//60), int(remaining%60)
    print(f"  [{lbl}] step {step:.0f}s | elapsed {h}h{m:02d}m{s:02d}s | ETA {r_h}h{r_m:02d}m{r_s:02d}s")
    _STEP_START = _time.time()
print("Environment setup: timing initialized")

In [ ]:
# Install dependencies
!pip install -q scikit-learn matplotlib seaborn pandas
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
print("Dependencies installed")

---
## 2. Clone Repository & Mount Drive

In [ ]:
_STEP_START = _time.time()
import os, sys
from pathlib import Path

REPO_DIR = Path("/content/deepfake-social-media-detector")
if not REPO_DIR.exists():
    !git clone https://github.com/Nico3783/deepfake-social-media-detector.git {REPO_DIR}
else:
    print("Repository already cloned")

sys.path.insert(0, str(REPO_DIR))
_T("Clone Repository")

In [ ]:
_STEP_START = _time.time()
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
# Mount the NEW Drive (the one with free space where you
# uploaded preprocessed_data.zip).
# When the popup appears, select the NEW account.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted")
_T("Mount Drive")

In [ ]:
_STEP_START = _time.time()
# ============================================================
# Google Drive API — authenticate with the SHARED FOLDER account
# ============================================================
# When the auth popup appears, select the account that owns
# the shared results folder AND has the preprocessed_data.zip
# ============================================================
from google.colab import auth as colab_auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

colab_auth.authenticate_user()
import google.auth
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_service = build('drive', 'v3', credentials=creds)

RESULTS_FOLDER_ID = "1rHxrAejBGqnBmvmrZsHcjvhM7Cwr7-KJ"

def get_or_create_subfolder(parent_id, name):
    query = f"name='{name}' and '{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
    results = drive_service.files().list(q=query, fields='files(id)').execute()
    if results['files']:
        return results['files'][0]['id']
    meta = {'name': name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]}
    folder = drive_service.files().create(body=meta, fields='id').execute()
    return folder['id']

# Verify which account is authenticated
about = drive_service.about().get(fields="user(emailAddress, quotaUsage, quotaLimit)").execute()
email = about['user']['emailAddress']
used_gb = int(about['user'].get('quotaUsage', 0)) / (1024**3)
limit_gb = int(about['user'].get('quotaLimit', 0)) / (1024**3)
print(f"Authenticated as: {email}")
print(f"Storage: {used_gb:.1f} GB / {limit_gb:.1f} GB")

print(f"Results folder: {RESULTS_FOLDER_ID}")
_T("Drive API")

In [ ]:
_STEP_START = _time.time()
import io as _io, json as _json

def save_json_checkpoint(name, data):
    ckpt_folder = get_or_create_subfolder(RESULTS_FOLDER_ID, "checkpoints")
    local = Path(f"/content/_ckpt_{name}.json")
    with open(local, "w") as f:
        _json.dump(data, f, indent=2, default=str)
    query = f"name='{name}.json' and '{ckpt_folder}' in parents and trashed=false"
    existing = drive_service.files().list(q=query, fields='files(id)').execute().get('files', [])
    if existing:
        drive_service.files().update(fileId=existing[0]['id'], media_body=MediaFileUpload(str(local), mimetype='application/json')).execute()
    else:
        drive_service.files().create(body={'name': f'{name}.json', 'parents': [ckpt_folder]},
            media_body=MediaFileUpload(str(local), mimetype='application/json'), fields='id').execute()
    local.unlink(missing_ok=True)
    print(f"  [Drive checkpoint] {name} saved")

def load_json_checkpoint(name):
    ckpt_folder_id = None
    query = f"name='checkpoints' and '{RESULTS_FOLDER_ID}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
    results = drive_service.files().list(q=query, fields='files(id)').execute()
    if results['files']:
        ckpt_folder_id = results['files'][0]['id']
    if not ckpt_folder_id:
        return None
    query = f"name='{name}.json' and '{ckpt_folder_id}' in parents and trashed=false"
    files = drive_service.files().list(q=query, fields='files(id, name)').execute().get('files', [])
    if not files:
        return None
    local = Path(f"/content/_ckpt_{name}.json")
    request = drive_service.files().get_media(fileId=files[0]['id'])
    with open(local, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    with open(local) as f:
        data = _json.load(f)
    local.unlink(missing_ok=True)
    print(f"  [Drive checkpoint] {name} loaded from Drive")
    return data

def save_fig_to_drive(fig, name):
    fig_folder = get_or_create_subfolder(RESULTS_FOLDER_ID, "figures")
    local = Path(f"/content/_fig_{name}.png")
    fig.savefig(local, dpi=150, bbox_inches="tight")
    query = f"name='{name}.png' and '{fig_folder}' in parents and trashed=false"
    existing = drive_service.files().list(q=query, fields='files(id)').execute().get('files', [])
    if existing:
        drive_service.files().update(fileId=existing[0]['id'], media_body=MediaFileUpload(str(local), mimetype='image/png')).execute()
    else:
        drive_service.files().create(body={'name': f'{name}.png', 'parents': [fig_folder]},
            media_body=MediaFileUpload(str(local), mimetype='image/png'), fields='id').execute()
    local.unlink(missing_ok=True)
    print(f"  [Drive figure] {name}.png saved")

print("Checkpoint helpers ready")
_T("Checkpoint Helpers")

---
## 3. Fetch Preprocessed Data from Drive

In [ ]:
_STEP_START = _time.time()
import zipfile

LOCAL_DATA_DIR = Path("/content/data")
PROCESSED_DIR = LOCAL_DATA_DIR / "processed"

# Check if already extracted locally
if PROCESSED_DIR.exists() and any(PROCESSED_DIR.glob('faces/**/*.jpg')):
    face_count = len(list(PROCESSED_DIR.glob('faces/**/*.jpg')))
    print(f"Already extracted: {face_count} face images on local disk")
else:
    # Search for zip on Drive
    print("Searching for preprocessed_data.zip on Drive...")
    query = f"name='preprocessed_data.zip' and '{RESULTS_FOLDER_ID}' in parents and trashed=false"
    files = drive_service.files().list(q=query, fields='files(id, name, size)').execute().get('files', [])

    if not files:
        # Also check root of Drive
        query2 = f"name='preprocessed_data.zip' and trashed=false"
        files = drive_service.files().list(q=query2, fields='files(id, name, size)').execute().get('files', [])

    if files:
        f = files[0]
        size_mb = int(f.get('size', 0)) / (1024*1024)
        print(f"Found: {f['name']} ({size_mb:.1f} MB)")
        print("Downloading... this may take a few minutes")

        local_zip = Path("/content/preprocessed_data.zip")
        request = drive_service.files().get_media(fileId=f['id'])
        with open(local_zip, "wb") as out:
            downloader = MediaIoBaseDownload(out, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"  {int(status.progress() * 100)}%", end="\r")
        print(f"\nExtracting...")
        with zipfile.ZipFile(local_zip, 'r') as z:
            z.extractall("/content/data")
        local_zip.unlink(missing_ok=True)
        face_count = len(list(PROCESSED_DIR.glob('faces/**/*.jpg')))
        print(f"Ready: {face_count} face images")
    else:
        print("ERROR: preprocessed_data.zip not found on Drive!")
        print("Please upload it to your Drive first.")

_T("Fetch Preprocessed Data")

In [ ]:
_STEP_START = _time.time()
# Verify data structure
for subdir in ["faces/ffpp_real", "faces/ffpp_fake", "faces/celeb_real", "faces/celeb_fake"]:
    path = PROCESSED_DIR / subdir
    if path.exists():
        count = len(list(path.glob("*.jpg")))
        print(f"  {subdir}: {count} images")
    else:
        print(f"  {subdir}: MISSING!")

# Verify metadata CSVs
METADATA_DIR = PROCESSED_DIR / "metadata"
for csv_name in ["all_metadata.csv", "train.csv", "val.csv", "test.csv"]:
    csv_path = METADATA_DIR / csv_name
    if csv_path.exists():
        import csv
        with open(csv_path) as f:
            rows = list(csv.DictReader(f))
        print(f"  {csv_name}: {len(rows)} rows")
    else:
        print(f"  {csv_name}: MISSING!")

_T("Verify Data")

---
## 4. Load Splits & Metadata

In [ ]:
_STEP_START = _time.time()
import csv, json, random
from collections import defaultdict

# Try loading from Drive checkpoint first
split_ckpt = load_json_checkpoint("metadata_splits")
if split_ckpt:
    train_rows = split_ckpt["train_rows"]
    val_rows = split_ckpt["val_rows"]
    test_rows = split_ckpt["test_rows"]
    all_metadata = split_ckpt["all_metadata"]
    print(f"Loaded splits from Drive checkpoint: {len(all_metadata)} total images")
else:
    # Load from CSVs
    def load_csv(path):
        rows = []
        if path.exists():
            with open(path) as f:
                for r in csv.DictReader(f):
                    r["label"] = int(r["label"])
                    rows.append(r)
        return rows

    train_rows = load_csv(METADATA_DIR / "train.csv")
    val_rows = load_csv(METADATA_DIR / "val.csv")
    test_rows = load_csv(METADATA_DIR / "test.csv")
    all_metadata = load_csv(METADATA_DIR / "all_metadata.csv")

    # Save checkpoint to Drive
    save_json_checkpoint("metadata_splits", {
        "train_rows": train_rows, "val_rows": val_rows,
        "test_rows": test_rows, "all_metadata": all_metadata,
    })

print(f"Total face images: {len(all_metadata)}")
print(f"  Train: {len(train_rows)}  |  Val: {len(val_rows)}  |  Test: {len(test_rows)}")
print(f"  Real: {sum(1 for r in all_metadata if r['label'] == 0)}")
print(f"  Fake: {sum(1 for r in all_metadata if r['label'] == 1)}")

_T("Load Splits")

---
## 5. Dataset & DataLoader Setup

In [ ]:
_STEP_START = _time.time()
import sys
sys.path.insert(0, str(REPO_DIR))

from torch.utils.data import DataLoader
from torchvision import transforms

# Dataset class
from PIL import Image

class DeepfakeDataset:
    def __init__(self, metadata, transform=None):
        self.metadata = metadata
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata[idx]
        image = Image.open(row["video_path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = row["label"]
        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

_T("Dataset Setup")

In [ ]:
_STEP_START = _time.time()
BATCH_SIZE = 64
NUM_WORKERS = 2

train_dataset = DeepfakeDataset(train_rows, transform=train_transform)
val_dataset = DeepfakeDataset(val_rows, transform=val_transform)
test_dataset = DeepfakeDataset(test_rows, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")
_T("DataLoader")

---
## 6. Load Models

In [ ]:
_STEP_START = _time.time()
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Load XceptionNet
from models.xception import create_xception_model
xception_model = create_xception_model(num_classes=2, pretrained=True)
xception_model = xception_model.to(DEVICE)

# Load EfficientNet-B0
from models.efficientnet import create_efficientnet_model
efficientnet_model = create_efficientnet_model(num_classes=2, pretrained=True)
efficientnet_model = efficientnet_model.to(DEVICE)

print(f"XceptionNet: {sum(p.numel() for p in xception_model.parameters()):,} params")
print(f"EfficientNet-B0: {sum(p.numel() for p in efficientnet_model.parameters()):,} params")
_T("Load Models")

---
## 7. Training Functions

In [ ]:
_STEP_START = _time.time()
import time
from datetime import datetime

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def train_model(model, train_loader, val_loader, device, model_name,
                num_epochs=30, learning_rate=1e-4, early_stopping_patience=7):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0

    # Resume from Drive checkpoint if available
    ckpt = load_json_checkpoint(f"{model_name}_training_state")
    if ckpt and ckpt.get("epoch", 0) > 0:
        start_epoch = ckpt["epoch"]
        best_val_loss = ckpt["best_val_loss"]
        best_epoch = ckpt["best_epoch"]
        patience_counter = ckpt.get("patience_counter", 0)
        history = ckpt.get("history", history)
        print(f"Resuming {model_name} from epoch {start_epoch} (best_loss={best_val_loss:.4f})")
    else:
        start_epoch = 0

    for epoch in range(start_epoch, num_epochs):
        # Train
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            train_correct += predicted.eq(labels).sum().item()
            train_total += labels.size(0)

        # Validate
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                val_total += labels.size(0)

        train_loss /= train_total
        val_loss /= val_total
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        scheduler.step(val_loss)

        print(f"Epoch {epoch+1}/{num_epochs} — "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(), OUTPUT_DIR / f"{model_name}_best.pth")
            # Save checkpoint to Drive
            save_json_checkpoint(f"{model_name}_training_state", {
                "epoch": epoch + 1, "best_val_loss": best_val_loss,
                "best_epoch": best_epoch, "patience_counter": patience_counter,
                "history": history,
            })
        else:
            patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Save history
    with open(OUTPUT_DIR / f"{model_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"Best: epoch {best_epoch+1}, val_loss={best_val_loss:.4f}")
    return history, best_val_loss

_T("Training Functions")

---
## 8. Train XceptionNet

In [ ]:
_STEP_START = _time.time()
xception_history, xception_best_loss = train_model(
    xception_model, train_loader, val_loader, DEVICE,
    model_name="xception", num_epochs=30, learning_rate=1e-4
)
_T("Train XceptionNet")

---
## 9. Train EfficientNet-B0

In [ ]:
_STEP_START = _time.time()
efficientnet_history, efficientnet_best_loss = train_model(
    efficientnet_model, train_loader, val_loader, DEVICE,
    model_name="efficientnet", num_epochs=30, learning_rate=1e-4
)
_T("Train EfficientNet")

---
## 10. Training Visualization

In [ ]:
_STEP_START = _time.time()
import matplotlib.pyplot as plt

def plot_training_curves(history_path, model_name):
    with open(history_path) as f:
        history = json.load(f)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train", linewidth=2)
    axes[0].plot(history["val_loss"], label="Validation", linewidth=2)
    axes[0].set_title(f"{model_name} - Loss", fontsize=14)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(history["train_acc"], label="Train", linewidth=2)
    axes[1].plot(history["val_acc"], label="Validation", linewidth=2)
    axes[1].set_title(f"{model_name} - Accuracy", fontsize=14)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    save_path = OUTPUT_DIR / f"{model_name}_training_curves.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    save_fig_to_drive(fig, f"{model_name}_training_curves")

plot_training_curves(OUTPUT_DIR / "xception_history.json", "XceptionNet")
plot_training_curves(OUTPUT_DIR / "efficientnet_history.json", "EfficientNet-B0")
_T("Training Curves")

---
## 11. Test Set Evaluation

In [ ]:
_STEP_START = _time.time()
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    start_time = time.time()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    elapsed = time.time() - start_time
    return {
        "accuracy": float(accuracy_score(all_labels, all_preds)),
        "precision": float(precision_score(all_labels, all_preds)),
        "recall": float(recall_score(all_labels, all_preds)),
        "f1": float(f1_score(all_labels, all_preds)),
        "auroc": float(roc_auc_score(all_labels, all_probs)),
        "samples_per_second": len(all_labels) / elapsed,
    }

_T("Eval Functions")

In [ ]:
_STEP_START = _time.time()

eval_ckpt = load_json_checkpoint("test_evaluation")
if eval_ckpt:
    results = eval_ckpt
    print("Test evaluation loaded from Drive checkpoint")
else:
    results = {}
    for model_name, model in [("XceptionNet", xception_model), ("EfficientNet-B0", efficientnet_model)]:
        print(f"Evaluating {model_name}...")
        metrics = evaluate_model(model, test_loader, DEVICE)
        results[model_name] = metrics
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    with open(OUTPUT_DIR / "test_evaluation.json", "w") as f:
        json.dump(results, f, indent=2)
    save_json_checkpoint("test_evaluation", results)

_T("Test Evaluation")

---
## 12. Deployment Assessment

In [ ]:
_STEP_START = _time.time()

def measure_inference_speed(model, input_size, num_iterations=100, batch_size=1):
    model.eval()
    dummy = torch.randn(batch_size, 3, input_size, input_size).to(DEVICE)
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            model(dummy)
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(num_iterations):
        with torch.no_grad():
            model(dummy)
    torch.cuda.synchronize()
    elapsed = time.time() - start
    avg_time = elapsed / num_iterations
    return {
        "fps": batch_size / avg_time,
        "avg_time_per_batch_ms": avg_time * 1000,
        "memory_mb": torch.cuda.max_memory_allocated() / (1024**2),
    }

deploy_ckpt = load_json_checkpoint("deployment_assessment")
if deploy_ckpt:
    deployment_results = deploy_ckpt
    print("Deployment assessment loaded from Drive checkpoint")
else:
    deployment_results = {}
    for model_name, model, input_size in [
        ("XceptionNet", xception_model, 299),
        ("EfficientNet-B0", efficientnet_model, 224),
    ]:
        print(f"Profiling {model_name}...")
        torch.cuda.reset_peak_memory_stats()
        speed = measure_inference_speed(model, input_size)
        param_count = sum(p.numel() for p in model.parameters())
        deployment_results[model_name] = {
            "total_parameters": param_count,
            "model_size_mb": round(param_count * 4 / (1024**2), 2),
            "input_size": input_size,
            "inference_speed": speed,
        }
        print(f"  Params: {param_count:,} | Size: {param_count * 4 / (1024**2):.2f} MB")
        print(f"  FPS: {speed['fps']:.2f} | Latency: {speed['avg_time_per_batch_ms']:.2f} ms")

    with open(OUTPUT_DIR / "deployment_assessment.json", "w") as f:
        json.dump(deployment_results, f, indent=2)
    save_json_checkpoint("deployment_assessment", deployment_results)

_T("Deployment Assessment")

---
## 13. Confusion Matrix

In [ ]:
_STEP_START = _time.time()
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, test_loader, model_name, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"], yticklabels=["Real", "Fake"], ax=ax)
    ax.set_title(f"{model_name} - Confusion Matrix", fontsize=14)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.tight_layout()
    save_path = OUTPUT_DIR / f"{model_name}_confusion_matrix.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    save_fig_to_drive(fig, f"{model_name}_confusion_matrix")

plot_confusion_matrix(xception_model, test_loader, "XceptionNet", DEVICE)
plot_confusion_matrix(efficientnet_model, test_loader, "EfficientNet-B0", DEVICE)
_T("Confusion Matrices")

---
## 14. ROC Curves

In [ ]:
_STEP_START = _time.time()
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(models_dict, test_loader, device):
    fig, ax = plt.subplots(figsize=(8, 6))
    for model_name, model in models_dict.items():
        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                probs = torch.softmax(outputs, dim=1)[:, 1]
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.numpy())
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {roc_auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
    ax.set_xlabel("False Positive Rate", fontsize=12)
    ax.set_ylabel("True Positive Rate", fontsize=12)
    ax.set_title("ROC Curve Comparison", fontsize=14)
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_path = OUTPUT_DIR / "roc_comparison.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    save_fig_to_drive(fig, "roc_comparison")

plot_roc_curves({"XceptionNet": xception_model, "EfficientNet-B0": efficientnet_model}, test_loader, DEVICE)
_T("ROC Curves")

---
## 15. Model Comparison

In [ ]:
_STEP_START = _time.time()

def plot_model_comparison(results, save_dir):
    metrics = ["accuracy", "precision", "recall", "f1", "auroc"]
    model_names = list(results.keys())
    x = np.arange(len(metrics))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, model_name in enumerate(model_names):
        values = [results[model_name].get(m, 0) for m in metrics]
        bars = ax.bar(x + i * width, values, width, label=model_name)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2., bar.get_height(),
                    f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Score")
    ax.set_title("Model Comparison - Test Set Performance", fontsize=14)
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([m.upper() for m in metrics])
    ax.legend()
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    save_path = save_dir / "model_comparison.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    save_fig_to_drive(fig, "model_comparison")

plot_model_comparison(results, OUTPUT_DIR)
_T("Model Comparison")

---
## 16. Save All Metrics

In [ ]:
_STEP_START = _time.time()

all_results = {
    "test_evaluation": results,
    "deployment_assessment": deployment_results,
    "metadata": {
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "test_samples": len(test_dataset),
        "seed": 42,
        "device": str(DEVICE),
    },
}

with open(OUTPUT_DIR / "all_experiment_results.json", "w") as f:
    json.dump(all_results, f, indent=2, default=str)

# Export CSV
csv_path = OUTPUT_DIR / "thesis_metrics.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Model", "Accuracy", "Precision", "Recall", "F1", "AUROC", "FPS", "Latency_ms", "Params", "Size_MB"])
    for model_name in results:
        r = results[model_name]
        d = deployment_results[model_name]
        writer.writerow([
            model_name, r["accuracy"], r["precision"], r["recall"], r["f1"], r["auroc"],
            d["inference_speed"]["fps"], d["inference_speed"]["avg_time_per_batch_ms"],
            d["total_parameters"], d["model_size_mb"],
        ])

save_json_checkpoint("all_experiment_results", all_results)
print(f"Saved: {OUTPUT_DIR / 'all_experiment_results.json'}")
print(f"Saved: {csv_path}")
_T("Save All Metrics")

---
## 17. Upload Results to Drive

In [ ]:
_STEP_START = _time.time()
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

def upload_to_shared_folder(local_dir, folder_id, subfolder_name):
    folder_metadata = {
        'name': subfolder_name,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [folder_id]
    }
    folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
    new_folder_id = folder['id']
    print(f"Created folder: {subfolder_name}")

    uploaded = 0
    for item in local_dir.rglob("*"):
        if item.is_file():
            rel_path = str(item.relative_to(local_dir))
            parts = Path(rel_path).parts
            parent_id = new_folder_id
            for part in parts[:-1]:
                query = f"name='{part}' and '{parent_id}' in parents and mimeType='application/vnd.google-apps.folder' and trashed=false"
                res = drive_service.files().list(q=query, fields='files(id)').execute()
                if res['files']:
                    parent_id = res['files'][0]['id']
                else:
                    sub = drive_service.files().create(
                        body={'name': part, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [parent_id]},
                        fields='id').execute()
                    parent_id = sub['id']

            file_metadata = {'name': item.name, 'parents': [parent_id]}
            media = MediaFileUpload(str(item), resumable=True)
            drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
            uploaded += 1
            print(f"  Uploaded: {rel_path}")

    print(f"Uploaded {uploaded} files to shared folder")

subfolder_name = f"experiment_results_{timestamp}"
upload_to_shared_folder(OUTPUT_DIR, RESULTS_FOLDER_ID, subfolder_name)
_T("Upload Results")

---
## Summary

### What was done:
1. Mounted new Google Drive and authenticated Drive API
2. Downloaded and extracted preprocessed_data.zip from Drive
3. Verified face images and metadata CSVs
4. Loaded train/val/test splits
5. Created DataLoaders with optimized settings
6. Trained XceptionNet and EfficientNet-B0 with AMP
7. Evaluated both models on test set
8. Generated confusion matrices, ROC curves, model comparison
9. Saved all metrics as JSON + CSV
10. Uploaded all results to shared Drive folder

### Checkpoints saved to Drive:
- `metadata_splits.json` — train/val/test splits
- `xception_training_state.json` — training resume state
- `efficientnet_training_state.json` — training resume state
- `test_evaluation.json` — model metrics
- `deployment_assessment.json` — speed/size
- `all_experiment_results.json` — everything combined
- Training curves, confusion matrices, ROC as PNGs